# Practical Cryptography for Application Security

**Module:** Securing Applications  
**Topic:** Cryptography for Securing Applications

---

## Learning Objectives

By the end of this session you will be able to:

1. Explain the difference between encoding, encryption, and hashing
2. Demonstrate the avalanche effect and explain why it matters
3. Identify weaknesses in naive hashing schemes
4. Hash passwords securely using salting and key-stretching
5. Encrypt and decrypt data symmetrically with AES
6. Encrypt data asymmetrically with RSA and explain when each approach is appropriate
7. Apply these techniques to real coursework scenarios

---

## Core Security Properties

| Property | Mechanism | Goal |
|---|---|---|
| **Confidentiality** | Encryption | Only authorised parties can read the data |
| **Integrity** | Hashing | Detect unauthorised modification |
| **Authentication** | Passwords / Signatures | Verify identity |

### Encoding ≠ Encryption ≠ Hashing

These are frequently confused — they are **not interchangeable**:

- **Encoding** (e.g. Base64, URL-encoding) — transforms data into a different *representation*. No secret key. Trivially reversible. Provides **zero** security.
- **Encryption** — transforms data using a *secret key*. Reversible only with the key. Provides **confidentiality**.
- **Hashing** — one-way transformation. Cannot be reversed. Used for **integrity** and **password storage**.

---

## Setup

**Sections 1 and 2** use only the Python standard library (`hashlib`, `hmac`, `os`, `secrets`) — no installation required.

**Sections 3–6** use the [PyCA `cryptography`](https://cryptography.io) package, which wraps OpenSSL and is the industry-standard Python cryptography library. It is a transitive dependency of many major packages (requests, paramiko, AWS SDKs) and receives broad security scrutiny. Run the cell below once if it is not already installed.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "cryptography", "-q"])
print("Dependencies ready.")

Dependencies ready.


---

## Section 1 — Hashing and Data Integrity

### Theory

A **cryptographic hash function** has three essential properties:

1. **One-way** — given a hash digest, it is computationally infeasible to recover the original input.
2. **Avalanche effect** — a tiny change in input produces a completely different output. Even flipping a single bit should change ~50 % of the digest bits.
3. **Collision resistance** — it should be computationally infeasible to find two *different* inputs that produce the *same* digest.

These properties make hashes useful for **verifying that data has not been tampered with**.

In [2]:
# stdlib only — no third-party packages needed for hashing
import hashlib

# A payment instruction — the kind of string you might sign or verify server-side.
message = "Transfer 250 EUR to account 458923"

digest = hashlib.sha256(message.encode()).hexdigest()

print("Message :", message)
print("SHA-256 :", digest)
print("Length  :", len(digest), "hex characters =", len(digest) * 4, "bits")

Message : Transfer 250 EUR to account 458923
SHA-256 : 3a93f9d9886fdc422986a9ff63a982a3abc07367428ce6d8e4ef7196f67d8c2e
Length  : 64 hex characters = 256 bits


### 1.1 — Avalanche Effect

Change the amount from **250** to **2500** — a single extra character — and observe the result.

In [3]:
msg1 = "Transfer 250 EUR to account 458923"
msg2 = "Transfer 2500 EUR to account 458923"  # one extra digit

hash1 = hashlib.sha256(msg1.encode()).hexdigest()
hash2 = hashlib.sha256(msg2.encode()).hexdigest()

print("Message 1 :", msg1)
print("Hash    1 :", hash1)
print()
print("Message 2 :", msg2)
print("Hash    2 :", hash2)
print()

# Count how many bit positions differ between the two digests
bits1 = bin(int(hash1, 16))[2:].zfill(256)
bits2 = bin(int(hash2, 16))[2:].zfill(256)
differing_bits = sum(b1 != b2 for b1, b2 in zip(bits1, bits2))

print(f"Bits that differ: {differing_bits} / 256  ({differing_bits / 256 * 100:.1f} %)")
print()
print("Even though the messages share almost identical content,")
print("the hashes look completely unrelated — that is the avalanche effect.")

Message 1 : Transfer 250 EUR to account 458923
Hash    1 : 3a93f9d9886fdc422986a9ff63a982a3abc07367428ce6d8e4ef7196f67d8c2e

Message 2 : Transfer 2500 EUR to account 458923
Hash    2 : 35a88516f70f5ff4f166929fd1550a844e650d3f52114783b656d25a0248c673

Bits that differ: 132 / 256  (51.6 %)

Even though the messages share almost identical content,
the hashes look completely unrelated — that is the avalanche effect.


> **Try it yourself:** Edit `msg2` above — change a single letter, a space, or the account number. Re-run the cell and observe how the hash changes completely each time.

---

### 1.2 — Weak Hash: Collision Demonstration

Not all hash functions are equal. Consider a naive scheme that simply sums the ASCII values of each character:

$$H(x) = \left(\sum_{i=1}^{n} \text{ASCII}(x_i)\right) \bmod 256$$

This is **not** a cryptographic hash. It is vulnerable to **collisions**: two different inputs can produce the same output.

In [4]:
def weak_hash(input_str: str) -> int:
    """Sum of ASCII values mod 256 — illustrative only, NOT secure."""
    return sum(ord(c) for c in input_str) % 256


# These two strings are anagrams, so their ASCII sums are identical.
word_a = "SECURITY"
word_b = "YTRUICES"  # same letters, different order

print(f"weak_hash('{word_a}') = {weak_hash(word_a)}")
print(f"weak_hash('{word_b}') = {weak_hash(word_b)}")
print()
print("Same hash — COLLISION. An attacker can substitute one for the other.")

weak_hash('SECURITY') = 120
weak_hash('YTRUICES') = 120

Same hash — COLLISION. An attacker can substitute one for the other.


Now compare with SHA-256 on the same inputs.

In [5]:
for word in ["SECURITY", "YTRUICES"]:
    digest = hashlib.sha256(word.encode()).hexdigest()
    print(f"SHA-256('{word}') = {digest}")

print()
print("SHA-256 produces completely different digests for anagram inputs.")
print("The weak hash cannot distinguish them at all.")

SHA-256('SECURITY') = c3a409ae6f7ebfb40817925a51de04278ebbb00c0ee492b0265091914d6af9d3
SHA-256('YTRUICES') = 5046359e3d0725484fc9922eb5806107bd9b92dae0791b854145c17061af859c

SHA-256 produces completely different digests for anagram inputs.
The weak hash cannot distinguish them at all.


> **Why do collisions matter?**  
> If an attacker can find a second input that produces the same hash as a legitimate one, they can substitute forged data while integrity checks still pass. This is a fundamental attack on data integrity.

---

## Section 2 — Password Security

### The Problem with Plaintext and Fast Hashes

Storing passwords as plaintext is obviously dangerous — a single database breach exposes every user. But even storing a simple SHA-256 hash is insufficient:

- **Rainbow tables** — precomputed tables of hash → password mappings for common passwords
- **GPU cracking** — SHA-256 is *designed* to be fast; a modern GPU can compute billions of SHA-256 hashes per second

### Solutions

1. **Salting** — append a random, unique value to the password before hashing. Defeats rainbow tables.
2. **Key stretching** — iterate the hash thousands of times. Makes brute-force dramatically slower. PBKDF2 and bcrypt both do this.

### ❌ Insecure — Plaintext Storage

In [6]:
# DO NOT do this in any real application.
password = "mypassword123"
stored_password = password  # stored directly in the database

print("Stored value:", stored_password)
print()
print("If the database is breached, every password is immediately visible.")

Stored value: mypassword123

If the database is breached, every password is immediately visible.


### ✅ Secure — Salted Hash with PBKDF2

In [7]:
import os

password = "mypassword123"

# Generate a random 16-byte salt — unique per user, stored alongside the hash.
salt = os.urandom(16)

# PBKDF2-HMAC: apply SHA-256 100,000 times.
# 100,000 iterations means an attacker must do 100,000x more work per guess.
hash_bytes = hashlib.pbkdf2_hmac(
    hash_name='sha256',
    password=password.encode(),
    salt=salt,
    iterations=100_000
)

print("Salt (hex):", salt.hex())
print("Hash (hex):", hash_bytes.hex())
print()
print("Both the salt and hash are stored in the database.")
print("The plaintext password is NEVER stored.")

Salt (hex): f852ef9030d670de2c0caef2595b710a
Hash (hex): 126f5b9c41a2a1497d7cdbed7e68d1aecc581d4138f2d299576fbe9cd1639f2e

Both the salt and hash are stored in the database.
The plaintext password is NEVER stored.


### Password Verification

To verify a login attempt, re-hash the supplied password with the stored salt and compare the result.

In [8]:
import hmac as hmac_mod  # stdlib — constant-time comparison


def hash_password(password: str) -> tuple[bytes, bytes]:
    """Hash a password. Returns (hash, salt) — store both."""
    salt = os.urandom(16)
    pw_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    return pw_hash, salt


def verify_password(candidate: str, stored_hash: bytes, salt: bytes) -> bool:
    """Return True if candidate matches the stored hash.

    hmac.compare_digest performs a constant-time comparison, preventing
    timing side-channel attacks that could leak information about the hash.
    """
    candidate_hash = hashlib.pbkdf2_hmac('sha256', candidate.encode(), salt, 100_000)
    return hmac_mod.compare_digest(candidate_hash, stored_hash)


# --- Simulate registration ---
user_password = "mypassword123"
stored_hash, salt = hash_password(user_password)
print("[Registration] Password hashed and stored.")

# --- Simulate correct login ---
result = verify_password("mypassword123", stored_hash, salt)
print(f"[Login] Correct password → {'GRANTED' if result else 'DENIED'}")

# --- Simulate wrong password ---
result = verify_password("wrongpassword", stored_hash, salt)
print(f"[Login] Wrong password   → {'GRANTED' if result else 'DENIED'}")

[Registration] Password hashed and stored.
[Login] Correct password → GRANTED
[Login] Wrong password   → DENIED


> **Note:** In production Python applications, prefer the `bcrypt` or `argon2-cffi` libraries over raw PBKDF2. They handle salt generation, storage format, and iteration count in a single opaque string, reducing the risk of implementation errors.

---

## Section 3 — Symmetric Encryption (AES / Rijndael)

### Theory

Symmetric encryption uses the **same key** for both encryption and decryption.

- Fast and efficient — suitable for bulk data
- **AES** (Advanced Encryption Standard) is based on the **Rijndael** algorithm and is the global industry standard, used in TLS, filesystem encryption, and more
- Key sizes: 128-bit, 192-bit, or 256-bit

### Modes of Operation

AES is a *block cipher* — it encrypts 16-byte blocks. A **mode of operation** determines how consecutive blocks are chained:

- **ECB** — each block encrypted independently. **Never use in practice** — identical plaintext blocks produce identical ciphertext blocks.
- **CBC** (Cipher Block Chaining) — each block is XOR'd with the previous ciphertext block before encryption. Requires a random **Initialisation Vector (IV)**.

### Risks

- **Key exposure** — if the key is leaked, all encrypted data is compromised immediately
- **No integrity by default** — AES-CBC encrypts but does not detect tampering. Use AES-GCM or add an HMAC for authenticated encryption.

### Encryption

In [14]:
# PyCA cryptography — wraps OpenSSL, the same library used by your OS and browser.
import os
import base64
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding as sym_padding

# 16-byte key → AES-128
key = os.urandom(16)

# A fresh random IV is required for every encryption operation.
# The IV does not need to be secret — it is sent alongside the ciphertext.
iv = os.urandom(16)

data = b"Krish is here, Halleluia"

# AES requires the plaintext length to be a multiple of the block size (16 bytes).
# PKCS7 padding adds the necessary bytes.
padder = sym_padding.PKCS7(128).padder()
padded_data = padder.update(data) + padder.finalize()

cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
encryptor = cipher.encryptor()
ciphertext = encryptor.update(padded_data) + encryptor.finalize()

print("Plaintext :", data)
print("IV        :", iv.hex())
print("Ciphertext:",  base64.b64encode(ciphertext))
print("Key:", base64.b64encode(key))
print("The ciphertext is unreadable without the key.")

Plaintext : b'Krish is here, Halleluia'
IV        : 299f8e39398aed516494b677d1bc631f
Ciphertext: b'6vDqkjrZHuuZJZEDaXJGPN8Lxo/jIPFS3IstmyW+0ts='
Key: b'938ZN/vVzZss2CvYTiXtQw=='
The ciphertext is unreadable without the key.


### Decryption

In [15]:
# Decryption requires the same key AND the same IV used during encryption.
cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
decryptor = cipher.decryptor()
padded_plaintext = decryptor.update(ciphertext) + decryptor.finalize()

# Strip the PKCS7 padding to recover the original plaintext.
unpadder = sym_padding.PKCS7(128).unpadder()
plaintext = unpadder.update(padded_plaintext) + unpadder.finalize()

print("Decrypted:", plaintext)
print("Match    :", plaintext == data)

Decrypted: b'Krish is here, Halleluia'
Match    : True


> **Try it yourself:** Change one byte of `ciphertext` (e.g. `ciphertext = bytes([ciphertext[0] ^ 1]) + ciphertext[1:]`) and try to decrypt. What happens?

---

## Section 4 — Encrypting Query Strings

### Application: Preventing Query String Tampering

A common attack is **parameter tampering**: a user modifies URL query strings to manipulate prices, IDs, or permissions.

```
# Original (as sent by the server)
GET /checkout?itemId=123&price=999

# Tampered (modified by the attacker)
GET /checkout?itemId=123&price=1
```

Encrypting the query string with a server-side key means the client cannot read or modify its contents.

In [11]:
import base64


def encrypt_query(query_string: str, key: bytes) -> str:
    """
    Encrypt a query string with AES-CBC.
    Returns a base64-encoded string safe for use in a URL.
    The IV is prepended to the ciphertext before encoding.
    """
    iv = os.urandom(16)
    padder = sym_padding.PKCS7(128).padder()
    padded = padder.update(query_string.encode()) + padder.finalize()
    encryptor = Cipher(algorithms.AES(key), modes.CBC(iv)).encryptor()
    ciphertext = encryptor.update(padded) + encryptor.finalize()
    # Prepend IV so the receiver can decrypt without out-of-band communication.
    return base64.urlsafe_b64encode(iv + ciphertext).decode()


def decrypt_query(token: str, key: bytes) -> str:
    """
    Decrypt a token produced by encrypt_query.
    Returns the original query string.
    """
    raw = base64.urlsafe_b64decode(token)
    iv, ciphertext = raw[:16], raw[16:]
    decryptor = Cipher(algorithms.AES(key), modes.CBC(iv)).decryptor()
    padded_plaintext = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = sym_padding.PKCS7(128).unpadder()
    return (unpadder.update(padded_plaintext) + unpadder.finalize()).decode()


# Server generates the key once and keeps it secret.
server_key = os.urandom(16)

query = "itemId=123&price=999"

token = encrypt_query(query, server_key)
print("Original query :", query)
print("Encrypted token:", token)
print()

recovered = decrypt_query(token, server_key)
print("Recovered query:", recovered)
print("Integrity check:", recovered == query)

Original query : itemId=123&price=999
Encrypted token: aMueGLDWwZZEHSjgXqYcyYuU-jh0A8jp0W3DdiWxDvsxyDPzUQBRP55pwzkwOOM4

Recovered query: itemId=123&price=999
Integrity check: True


> **Discussion:** Even with encryption, could an attacker replay a previously captured token to reuse an old price? What additional control would prevent this?

---

## Section 5 — Asymmetric Encryption (RSA)

### Theory

Asymmetric encryption uses a **mathematically linked key pair**:

- **Public key** — shared freely. Used to *encrypt* data or *verify* a signature.
- **Private key** — kept secret by the owner. Used to *decrypt* data or *create* a signature.

The crucial property: **data encrypted with the public key can only be decrypted with the corresponding private key.**

### Use Cases

| Use case | Who encrypts | Who decrypts |
|---|---|---|
| Secure key exchange | Sender (with recipient's public key) | Recipient (with private key) |
| Digital signature | Signer (with private key) | Verifier (with public key) |

### Limitation

RSA is **computationally expensive** and can only encrypt data smaller than its key size. It is **not** used to encrypt large payloads directly. In practice (e.g. in TLS), RSA is used to securely exchange an AES session key, then AES encrypts the actual data.

### Key Generation

In [19]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding as asym_padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.serialization import Encoding, PublicFormat

# Generate a 2048-bit RSA key pair.
# In production, key generation happens once and the keys are stored securely.
print("Generating 2048-bit RSA key pair (this may take a moment)...")
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
public_key = private_key.public_key()
print("Key pair generated.")
print()

# Show what a public key looks like (safe to share).
pem = public_key.public_bytes(Encoding.PEM, PublicFormat.SubjectPublicKeyInfo)
print("Public key (PEM format):")
print(pem.decode())

Generating 2048-bit RSA key pair (this may take a moment)...
Key pair generated.

Public key (PEM format):
-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAxWzuzZ3NHUBmKEDeSQlC
pOES+seVoYlf+2G/u6TmtXla51uhTdf4Ozw8DhWOeFhWgEiIqY3IkL7FVuM6XhM8
9qIpEYH/j4TGYpfh0eljafnFZv8f9RruYWMym6g8Hf7Tq0leN6wVKwVAOTePo24H
YJba1+HF3iAOZ7MfymTGOPZnMEgV1X1MXd8H/gR0ru0/1dsOFC4RPlXNHfVbXgMD
u8y+2gyFHTxhTD2+DBEN67L6Gl6K7wv3aw58Zq6Tx2qU11Vfa1pG+kjLF5QaSbnj
gq3WEUx8gRtdWWYHV7IvLaoKx9R+fLrq9+QWd73RqSIE1L1yj/mU4APdrEfage2I
vQIDAQAB
-----END PUBLIC KEY-----



### Encryption with RSA

In [20]:
# Encrypt with the recipient's PUBLIC key.
# OAEP with SHA-256 is the secure padding scheme for RSA encryption.
oaep_padding = asym_padding.OAEP(
    mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(),
    label=None,
)

message = b"AES session key: supersecretkey!"
encrypted = public_key.encrypt(message, oaep_padding)

print("Plaintext :", message)
print()
print("Encrypted (hex):")
print(encrypted.hex())
print()
print("Only the holder of the private key can decrypt this.")

Plaintext : b'AES session key: supersecretkey!'

Encrypted (hex):
9651351c2db786ba1371e1775496b753403d4b8d53d02b7bd7f23b05788c5d4912d06d5daa522b337e48207cfef7f22aa2d93048ee8a52bc9f20ba71d26887aa7307d4e03ca08cbca8950e3964c8287a007c5fb8174b0f3169bcf00892d036b687a89e5ed9075251885c8ac96a1bff84f76e9407682a0714bb9ef86e86ace3354062b5826464f489baa92113d52dce8771442bd7777c6c0fc7c1a92059b9d4c6a31fd1ab23752d11628adef2314028bd2b677e139943536711bcb43b05ca982589b2a3cb588a894a40f22810adbfc2c2b7b87af11d3f34fe6c9f1c220c793aebcb67673aa6dcadc685f058f174973f164967a32f7166df51144cec2f333bfc30

Only the holder of the private key can decrypt this.


### Decryption with the Private Key

In [21]:
# Decrypt with the PRIVATE key — only the key owner can do this.
decrypted = private_key.decrypt(encrypted, oaep_padding)

print("Decrypted:", decrypted)
print("Match    :", decrypted == message)

Decrypted: b'AES session key: supersecretkey!'
Match    : True


> **Try it yourself:** Generate a second RSA key pair. Try to decrypt the ciphertext above using the new private key. What error do you get?

---

## Section 6 — AES vs RSA: When to Use Which

| Dimension | AES (Symmetric) | RSA (Asymmetric) |
|---|---|---|
| Speed | Very fast | Slow |
| Data size | Unlimited | Limited by key size |
| Key distribution | Hard — both parties need the same secret key | Easy — share the public key openly |
| Typical use | Bulk data encryption | Key exchange, digital signatures |

**Real-world pattern (used in TLS):**

1. Client encrypts a random AES session key with the server's RSA public key
2. Server decrypts it with its RSA private key
3. Both parties now share a secret AES key → all subsequent data is encrypted with AES

In [15]:
# Demonstrate the hybrid encryption pattern.

# ── Key generation (done once, server-side) ────────────────────────────────
server_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
server_public_key = server_private_key.public_key()

oaep = asym_padding.OAEP(
    mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(),
    label=None,
)

# ── Client side ───────────────────────────────────────────────────────────
# 1. Generate a fresh AES session key.
session_key = os.urandom(16)

# 2. Encrypt the session key with the server's RSA public key.
encrypted_session_key = server_public_key.encrypt(session_key, oaep)

# 3. Encrypt the actual payload with AES using that session key.
session_iv = os.urandom(16)
payload = b"This is the actual sensitive payload data."
padder = sym_padding.PKCS7(128).padder()
padded_payload = padder.update(payload) + padder.finalize()
encryptor = Cipher(algorithms.AES(session_key), modes.CBC(session_iv)).encryptor()
encrypted_payload = session_iv + encryptor.update(padded_payload) + encryptor.finalize()

print("[Client] Session key generated and encrypted with server's RSA public key.")
print("[Client] Payload encrypted with AES session key.")

# ── Server side ───────────────────────────────────────────────────────────
# 1. Decrypt the session key with the RSA private key.
recovered_session_key = server_private_key.decrypt(encrypted_session_key, oaep)

# 2. Decrypt the payload with the recovered AES session key.
iv = encrypted_payload[:16]
decryptor = Cipher(algorithms.AES(recovered_session_key), modes.CBC(iv)).decryptor()
padded_plaintext = decryptor.update(encrypted_payload[16:]) + decryptor.finalize()
unpadder = sym_padding.PKCS7(128).unpadder()
recovered_payload = unpadder.update(padded_plaintext) + unpadder.finalize()

print("[Server] RSA private key decrypted the session key.")
print("[Server] AES decrypted the payload.")
print()
print("Recovered payload:", recovered_payload)
print("Integrity check  :", recovered_payload == payload)

[Client] Session key generated and encrypted with server's RSA public key.
[Client] Payload encrypted with AES session key.
[Server] RSA private key decrypted the session key.
[Server] AES decrypted the payload.

Recovered payload: b'This is the actual sensitive payload data.'
Integrity check  : True


---

## Section 7 — Linking to Your Coursework

Your assignment requires you to fix two security weaknesses in a FastAPI application.

---

### Task 1 — Password Security

**Problem:** Passwords are stored as plaintext (or with a trivially weak hash).  
**Solution:** Replace storage with a salted PBKDF2 or bcrypt hash and implement secure verification.

The pattern below mirrors what you need to implement in your application:

In [16]:
# Illustrative: what a FastAPI user registration endpoint should do.
# stdlib only — hashlib, hmac, os.

def register_user(username: str, plaintext_password: str) -> dict:
    """
    Hash the password before storing it.
    Returns a record safe to persist to a database.
    """
    salt = os.urandom(16)
    pw_hash = hashlib.pbkdf2_hmac('sha256', plaintext_password.encode(), salt, 100_000)
    return {
        "username": username,
        "password_hash": pw_hash.hex(),
        "salt": salt.hex(),
        # plaintext_password is NEVER stored
    }


def login_user(username: str, candidate_password: str, stored_record: dict) -> bool:
    """Verify a login attempt against a stored record."""
    salt = bytes.fromhex(stored_record["salt"])
    stored_hash = bytes.fromhex(stored_record["password_hash"])
    candidate_hash = hashlib.pbkdf2_hmac('sha256', candidate_password.encode(), salt, 100_000)
    return hmac_mod.compare_digest(candidate_hash, stored_hash)


record = register_user("alice", "hunter2")
print("Stored record:", record)
print()
print("Login (correct) :", login_user("alice", "hunter2", record))
print("Login (wrong)   :", login_user("alice", "password123", record))

Stored record: {'username': 'alice', 'password_hash': '734a460fc2390507f2167833cd2fe398963a506bb55aec3d95ccc437f2dbbc52', 'salt': '29075c001fee1f31931b90d0e30e3c6c'}

Login (correct) : True
Login (wrong)   : False


---

### Task 2 — Query String Encryption

**Problem:** Sensitive parameters (item IDs, prices) are passed as plaintext query strings that users can modify.  
**Solution:** Encrypt the query string server-side before embedding it in a URL; decrypt and parse it inside the controller.

The `encrypt_query` / `decrypt_query` helpers from Section 4 are ready to use — integrate them into your route handlers:

In [17]:
# Illustrative: FastAPI route integration pattern.

# app_key would be loaded from an environment variable — never hardcoded.
app_key = os.urandom(16)


# ── In the route that generates the link ──────────────────────────────────
def build_checkout_url(item_id: int, price: float) -> str:
    params = f"itemId={item_id}&price={price}"
    token = encrypt_query(params, app_key)
    return f"/checkout?token={token}"


# ── In the checkout route handler ─────────────────────────────────────────
def handle_checkout(token: str) -> dict:
    try:
        params = decrypt_query(token, app_key)
        # parse params → e.g. urllib.parse.parse_qs(params)
        return {"status": "ok", "params": params}
    except Exception:
        # Decryption failure means the token was tampered with.
        return {"status": "error", "detail": "Invalid or tampered token"}


url = build_checkout_url(item_id=123, price=999.99)
print("Generated URL:", url)
print()

token = url.split("token=")[1]
print("Checkout handler result:", handle_checkout(token))
print()

# Attacker tries to submit a tampered token
print("Tampered token result   :", handle_checkout("AAAA" + token[4:]))

Generated URL: /checkout?token=SMR-zLfM84Hf0cB7VnQM1tBpeNC1idJOcGdjD0Ie_v0GQyn4UqfcZYtmeXBXvhD-

Checkout handler result: {'status': 'ok', 'params': 'itemId=123&price=999.99'}

Tampered token result   : {'status': 'error', 'detail': 'Invalid or tampered token'}


---

## Section 8 — Reflection Questions

Work through these individually or in groups. Be prepared to discuss your reasoning.

1. **SHA-256 and passwords** — SHA-256 is a strong hash, yet it is insufficient alone for password storage. Explain *specifically* why, referencing both offline attacks and GPU throughput.

2. **AES key compromise** — If an AES key is leaked, what is the impact? Can you limit the blast radius? What does the concept of *forward secrecy* have to do with this?

3. **Collision danger** — Describe a concrete real-world scenario in which a hash collision could be exploited by an attacker. What integrity guarantee is broken?

4. **RSA vs AES** — A developer suggests encrypting the entire database with RSA because "public/private keys are more secure than a single shared key." What technical and practical problems does this decision have?

5. **CBC and integrity** — AES-CBC encrypts data but provides no integrity guarantee. An attacker who can flip bits in the ciphertext will produce garbled-but-decryptable plaintext with *predictable* corruption. Why is this dangerous? What mode or construction solves it?

---

## Quick Reference

| Algorithm | Type | Key Size | Use Case |
|---|---|---|---|
| SHA-256 | Hash | — | Data integrity, digital signatures |
| SHA-512 | Hash | — | Data integrity, higher security margin |
| PBKDF2 | Key derivation | — | Password hashing (with salt + iterations) |
| bcrypt | Key derivation | — | Password hashing (built-in salt + cost factor) |
| AES-128 | Symmetric cipher | 128-bit | Bulk encryption |
| AES-256 | Symmetric cipher | 256-bit | Bulk encryption, higher security margin |
| RSA-2048 | Asymmetric cipher | 2048-bit | Key exchange, digital signatures |

---

*Remember: working code is not the same as secure code. Always reason about threat models, key management, and what an attacker with access to your ciphertext or database can infer.*